# PART IV: Clustering

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap

from sklearn.base import clone
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from sklearn.preprocessing import MinMaxScaler

## Importing data

In [ ]:
df = pd.read_parquet("advanced_feature_selected_data.parquet")
df.info()

## Data Pre-processing

Scale all continuous attributes to a 0 to 1 scale using MinMaxScaler

In [ ]:
continuous_columns = [
    "Start_Lat",
    "Start_Lng",
    "Distance(mi)",
    "Temperature(F)",
    "Wind_Chill(F)",
    "Humidity(%)",
    "Precipitation(in)",
    "Duration_Minutes",
]

scaler = MinMaxScaler().fit(df[continuous_columns])

Take 1% subset

In [ ]:
sample_df, _ = train_test_split(
    df,
    train_size=0.01,
    stratify=df["Severity"],
    random_state=42,
)

sample_df[continuous_columns] = scaler.transform(sample_df[continuous_columns])

X = sample_df.drop(columns=["Severity"])
y = sample_df["Severity"]

Drop boolean values

In [ ]:
X = X.drop(columns=X.select_dtypes(include=["bool", "boolean"]).columns)

X = X.astype("float32")

## Clustering

In [ ]:
def make_cluster_plot(model, embedding, ax, title=None):
    cluster_labels = np.asarray(model.labels_)
    noise = cluster_labels == -1
    cluster_ids, color_values = np.unique(cluster_labels[~noise], return_inverse=True)
    n_clusters = len(cluster_ids)

    if n_clusters:
        scatter = ax.scatter(
            embedding[~noise, 0],
            embedding[~noise, 1],
            c=color_values,
            cmap=plt.get_cmap("turbo", n_clusters),
            vmin=-0.5,
            vmax=n_clusters - 0.5,
            s=12,
            alpha=0.75,
        )

        colorbar = ax.figure.colorbar(scatter, ax=ax)
        step = max(1, int(np.ceil(n_clusters / 20)))
        ticks = np.arange(0, n_clusters, step)
        colorbar.set_ticks(ticks)
        colorbar.set_ticklabels([str(cluster_ids[t]) for t in ticks])
        colorbar.set_label("Cluster")

    # Noise points (DBSCAN label -1) are drawn separately in gray
    if noise.any():
        ax.scatter(
            embedding[noise, 0],
            embedding[noise, 1],
            c="gray",
            s=12,
            alpha=0.75,
            label="Noise (-1)",
        )
        ax.legend(loc="best")

    ax.set(
        xlabel="UMAP 1",
        ylabel="UMAP 2",
        title=title or "Clusters in UMAP space",
    )


def make_cluster_plot_panel(models, embedding, labels=None):
    models = list(models)
    if labels is None:
        labels = [None] * len(models)

    columns = min(2, len(models))
    rows = (len(models) + columns - 1) // columns
    fig, axes = plt.subplots(
        rows, columns, figsize=(8 * columns, 5 * rows), squeeze=False
    )

    for model, label, ax in zip(models, labels, axes.flat):
        make_cluster_plot(model, embedding, ax, label)

    for ax in axes.flat[len(models):]:
        ax.remove()

    fig.suptitle("Clustering assignments visualized with UMAP")
    fig.tight_layout()
    return fig, axes


def compare_clustering_parameter(X, estimator, parameter, values, embedding, return_time=False, **fixed_params):
    base_model = estimator(**fixed_params)
    values = list(values)

    models = []
    fit_times = []

    for value in values:
        model = clone(base_model).set_params(**{parameter: value})
        
        start_time = time.perf_counter()
        model.fit(X)
        end_time = time.perf_counter()
        
        models.append(model)
        fit_times.append(end_time - start_time)

    labels = [
        f"{type(base_model).__name__}: {parameter}={value}"
        for value in values
    ]

    # Return the times list if return_time is True
    if return_time:
        return make_cluster_plot_panel(models, embedding, labels), [l for l in labels], [m for m in models], fit_times

    return make_cluster_plot_panel(models, embedding, labels), [l for l in labels], [m for m in models]

### UMAP embedding
Fit once here and reused by every cluster plot below.

In [ ]:
embedding = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
).fit_transform(X)

### K-means Clustering

In [ ]:
k_values = range(1, 30)
sse = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=0, n_init=10).fit(X)
    sse.append(model.inertia_)

plt.figure(figsize=(10, 4))
plt.plot(k_values, sse, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("SSE (inertia)")
plt.title("K-means elbow plot")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

In [ ]:
cluster_plot_fig, labels, models = compare_clustering_parameter(
    X,
    KMeans,
    parameter="n_clusters",
    values=[2, 5, 8, 10],
    embedding=embedding,
    random_state = 0
)
plt.show()

In [ ]:
def get_results(labels, models):
    stats = []
    for label, model in zip(labels, models):
        pct_dict = (
            (pd.Series(model.labels_).value_counts(normalize=True) * 100)
            .round(1)
            .to_dict()
        )

        stats.append(
            {
                "Model": label,
                "Iterations": model.n_iter_,
                "% per Cluster": pct_dict,
                "SSE": model.inertia_
            }
        )

    return pd.DataFrame(stats)

results_df = get_results(labels, models)
print(results_df.to_string(index=False))

In [ ]:
def analyze_kmeans_anomalies(kmeans_model, X, percentile=99):
    scores = np.min(kmeans_model.transform(X), axis=1)
    
    threshold = np.percentile(scores, percentile)
    outliers = scores > threshold
    X_values = X.to_numpy()
    
    plt.figure(figsize=(8, 5))
    plt.scatter(X_values[~outliers, 0], X_values[~outliers, 1], c='blue', alpha=0.5, label='Normal')
    plt.scatter(X_values[outliers, 0], X_values[outliers, 1], c='red', marker='x', s=80, label='Outliers')
    plt.scatter(kmeans_model.cluster_centers_[:, 0], kmeans_model.cluster_centers_[:, 1], 
                c='black', marker='*', s=200, label='Centroids')
    plt.title(f"K-Means Anomalies (Top {100-percentile}%)")
    plt.legend()
    plt.show()

    print(f"Outliers Identified: {outliers.sum()} (out of {len(X)} total points)")
    print(f"Distance Threshold f(x): {threshold:.2f}\n")
    
    if outliers.any():
        n_show = min(10, int(outliers.sum()))
        print(f"First {n_show} outliers:")
        print(X[outliers].head(n_show).round(2).to_string())

    return outliers

def analyze_centroids(kmeans_model, X):
    centroids = pd.DataFrame(kmeans_model.cluster_centers_, columns=X.columns)
    centroids.index.name = 'Cluster'
    print("Cluster Centers (Average Feature Values):")
    print(centroids)
    print("\nFeature Variance Across Centroids:")
    print(centroids.var().sort_values(ascending=False))

label = labels[3]
model = models[3]

print(label)
outliers = analyze_kmeans_anomalies(model, X)
analyze_centroids(model, X)

In [ ]:
cluster_plot_fig, labels, models= compare_clustering_parameter(
    X,
    KMeans,
    parameter="n_init",
    values=[1, 5, 10, 20],
    embedding=embedding,
    random_state = 0
)
plt.show()

In [ ]:
results_df = get_results(labels, models)
print(results_df.to_string(index=False))

In [ ]:
print(labels[3])
outliers = analyze_kmeans_anomalies(models[3], X)

In [ ]:
cluster_plot_fig, labels, models= compare_clustering_parameter(
    X,
    KMeans,
    parameter="max_iter",
    values=[1, 10],
    embedding=embedding,
    random_state = 0
)
plt.show()

In [ ]:
results_df = get_results(labels, models)
print(results_df.to_string(index=False))

## Agglomerative Clustering

In [ ]:
connectivity = kneighbors_graph(X, n_neighbors=100, include_self=False)

cluster_plot_fig, labels, models, fit_times = compare_clustering_parameter(
        X,
        AgglomerativeClustering,
        parameter="linkage",
        values=['ward', 'average', 'complete', 'single'],
        return_time = True,
        embedding=embedding,
        connectivity = connectivity,
        n_clusters=8
    )
plt.show()

In [ ]:
def get_results(labels, models, fit_times, X):
    stats = []
    for label, model, fit_time in zip(labels, models, fit_times):
        pct_dict = (
            (pd.Series(model.labels_).value_counts(normalize=True) * 100)
            .round(1)
            .to_dict()
        )

        n_unique_labels = len(np.unique(model.labels_))
        
        sil_score = silhouette_score(X, model.labels_)
        sil_score = round(sil_score, 4)

        stats.append(
            {
                "Model": label,
                "# of Clusters": n_unique_labels,
                "% per Cluster": pct_dict,
                "Silhouette Score": sil_score,
                "Fit Time": fit_time
            }
        )

    return pd.DataFrame(stats)

results_df = get_results(labels, models, fit_times, X)
print(results_df.to_string(index=False))

## DBSCAN

In [ ]:
min_samples_values = [5, 10, 20, 50]
min_samples_values = [m for m in min_samples_values if m < len(X)]

fig, ax = plt.subplots(figsize=(8, 5))
all_k_distances = []

for min_samples in min_samples_values:
    distances, _ = NearestNeighbors(n_neighbors=min_samples).fit(X).kneighbors(X)
    k_distances = np.sort(distances[:, -1])
    all_k_distances.extend(k_distances)
    ax.plot(k_distances, label=f"min_samples={min_samples}")

ax.set(
    xlabel="Points sorted by distance",
    ylabel="Distance to k-th neighbor",
    title="DBSCAN k-distance curves",
)
ax.legend()
ax.grid(True)
plt.show()

In [ ]:
cluster_plot_fig, labels, models, times = compare_clustering_parameter(
    X,
    DBSCAN,
    parameter="algorithm",
    return_time = True,
    values=['auto', 'ball_tree', 'kd_tree', 'brute'],
    embedding=embedding,
)
plt.show()

In [ ]:

results_df = get_results(labels, models, times, X)
print(results_df.to_string(index=False))

In [ ]:
cluster_plot_fig, labels, models, times= compare_clustering_parameter(
    X,
    DBSCAN,
    return_time = True,
    parameter="eps",
    values=[0.2, 0.3, 0.4, 0.5],
    embedding=embedding,
)
plt.show()

In [ ]:
results_df = get_results(labels, models, times, X)
print(results_df.to_string(index=False))

In [ ]:
cluster_plot_fig, labels, models, times = compare_clustering_parameter(
    X,
    DBSCAN,
    parameter="min_samples",
    return_time=True,
    values=[10, 50, 200, 1000],
    embedding=embedding,
)
plt.show()

In [ ]:

results_df = get_results(labels, models, times, X)
print(results_df.to_string(index=False))